# LCT — RF-DETR Medium @640 — low-LR fine-tune from our best @576

Проверяем, даст ли Medium тот же выигрыш от увеличения resolution,
который мы уже увидели у Small.

Нужные Kaggle Inputs:
- `grigoriyyyy/lct-base-combined-dataset` или `grigoriyyy/lct-base-combined-dataset`
- output предыдущего `lct-rfdetr-medium-576`, где есть
  `checkpoint_best_ema.pth` / `checkpoint_best_total.pth`

Схема:
- стартуем из уже construction-trained Medium @576;
- меняем resolution на `640`;
- делаем **4 эпохи**;
- low LR, чтобы не испортить уже хороший detector;
- checkpoint каждую эпоху;
- никакого длинного post-training eval в этом notebook.

Почему 4 эпохи: @640 заметно тяжелее @576, и здесь важнее гарантированно
получить законченный trajectory до 12-часового Kaggle hard limit.


In [1]:
!nvidia-smi
!pip install -q "rfdetr[train]==1.10.1" pyyaml pillow

import os
import json
import time
import random
import shutil
from pathlib import Path

import numpy as np
import torch
import yaml
import rfdetr
from PIL import Image

print("rfdetr:", getattr(rfdetr, "__version__", "unknown"))
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

from rfdetr import RFDETRMedium


Fri Sep 25 10:58:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
SEED = 42
RESOLUTION = 640
EPOCHS = 4

OUTPUT_DIR = Path("/kaggle/working/rfdetr_medium_640_finetune")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print({
    "resolution": RESOLUTION,
    "epochs": EPOCHS,
    "seed": SEED,
    "output_dir": str(OUTPUT_DIR),
})


{'resolution': 640, 'epochs': 4, 'seed': 42, 'output_dir': '/kaggle/working/rfdetr_medium_640_finetune'}


In [3]:
# Locate frozen external construction_v1 dataset.
INPUT = Path("/kaggle/input")
yamls = list(INPUT.rglob("data.yaml"))

print("Found data.yaml files:")
for y in yamls:
    print(" ", y)

assert yamls, "No data.yaml found under /kaggle/input"

candidates = [y for y in yamls if "construction_v1" in y.as_posix().lower()]
DATA_YAML = candidates[0] if candidates else yamls[0]
DATA_ROOT = DATA_YAML.parent

with open(DATA_YAML, "r", encoding="utf-8") as f:
    original_data = yaml.safe_load(f)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
expected_counts = {"train": 19391, "val": 4609, "test": 1102}

for split in ("train", "val", "test"):
    split_path = DATA_ROOT / original_data[split]
    n = sum(
        1 for p in split_path.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )
    print(f"{split}: {n}")
    assert n == expected_counts[split], (split, n, expected_counts[split])

names_raw = original_data["names"]
if isinstance(names_raw, dict):
    CLASS_NAMES = [
        names_raw[i] if i in names_raw else names_raw[str(i)]
        for i in range(len(names_raw))
    ]
else:
    CLASS_NAMES = list(names_raw)

assert len(CLASS_NAMES) == 13
print("DATA_ROOT =", DATA_ROOT)
print("Classes:", CLASS_NAMES)


Found data.yaml files:
  /kaggle/input/notebooks/grigoriyyy/lct-rfdetr-medium-576/rfdetr_dataset/data.yaml
  /kaggle/input/datasets/grigoriyyy/lct-base-combined-dataset/construction_v1/data.yaml
train: 19391
val: 4609
test: 1102
DATA_ROOT = /kaggle/input/datasets/grigoriyyy/lct-base-combined-dataset/construction_v1
Classes: ['excavator', 'dump_truck', 'truck', 'loader', 'bulldozer', 'motor_grader', 'roller', 'concrete_mixer', 'telehandler', 'piling_machine', 'crane_manipulator', 'mobile_crane', 'tower_crane']


In [4]:
# Build RF-DETR-compatible YOLO runtime view.
# Images are symlinked; labels are copied, with exact duplicate rows removed.
RF_DATA = Path("/kaggle/working/rfdetr_dataset")

if RF_DATA.exists():
    shutil.rmtree(RF_DATA)

for split in ("train", "valid"):
    (RF_DATA / split / "images").mkdir(parents=True, exist_ok=True)
    (RF_DATA / split / "labels").mkdir(parents=True, exist_ok=True)

def build_split(source_split: str, target_split: str):
    src_images = DATA_ROOT / "images" / source_split
    src_labels = DATA_ROOT / "labels" / source_split

    image_count = 0
    duplicate_lines_removed = 0
    seen_names = set()

    images = sorted(
        p for p in src_images.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

    for img in images:
        rel = img.relative_to(src_images)
        flat_stem = "__".join(list(rel.parts[:-1]) + [img.stem])
        target_name = flat_stem + img.suffix.lower()

        assert target_name not in seen_names, f"Collision: {target_name}"
        seen_names.add(target_name)

        os.symlink(img, RF_DATA / target_split / "images" / target_name)
        image_count += 1

        src_label = src_labels / rel.with_suffix(".txt")
        dst_label = RF_DATA / target_split / "labels" / (flat_stem + ".txt")

        if not src_label.exists():
            dst_label.write_text("", encoding="utf-8")
            continue

        lines = [
            x.strip()
            for x in src_label.read_text(encoding="utf-8").splitlines()
            if x.strip()
        ]
        unique = list(dict.fromkeys(lines))
        duplicate_lines_removed += len(lines) - len(unique)

        dst_label.write_text(
            ("\n".join(unique) + "\n") if unique else "",
            encoding="utf-8",
        )

    return {
        "images": image_count,
        "duplicate_lines_removed": duplicate_lines_removed,
    }

train_build = build_split("train", "train")
val_build = build_split("val", "valid")

rf_yaml = {
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
    "nc": len(CLASS_NAMES),
    "train": "train/images",
    "val": "valid/images",
}

(RF_DATA / "data.yaml").write_text(
    yaml.safe_dump(rf_yaml, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

assert train_build["images"] == 19391
assert val_build["images"] == 4609

print("RF-DETR runtime view ready:")
print(" train:", train_build)
print(" valid:", val_build)
print(" RF_DATA:", RF_DATA)


RF-DETR runtime view ready:
 train: {'images': 19391, 'duplicate_lines_removed': 1}
 valid: {'images': 4609, 'duplicate_lines_removed': 0}
 RF_DATA: /kaggle/working/rfdetr_dataset


In [5]:
# Prefer best EMA from previous Medium @576 run.
preferred_names = [
    "checkpoint_best_ema.pth",
    "checkpoint_best_total.pth",
    "checkpoint_best_regular.pth",
]

START_CKPT = None

for filename in preferred_names:
    candidates = [
        p for p in Path("/kaggle/input").rglob(filename)
        if "rfdetr" in p.as_posix().lower()
        and "medium" in p.as_posix().lower()
        and "576" in p.as_posix().lower()
    ]

    if candidates:
        print(f"{filename} candidates:")
        for p in candidates:
            print(" ", p)

        START_CKPT = sorted(candidates, key=lambda p: len(p.as_posix()))[0]
        break

assert START_CKPT is not None, (
    "Could not find RF-DETR Medium @576 best checkpoint. "
    "Attach the previous lct-rfdetr-medium-576 output."
)

print("\nStarting checkpoint:", START_CKPT)


checkpoint_best_ema.pth candidates:
  /kaggle/input/notebooks/grigoriyyy/lct-rfdetr-medium-576/rfdetr_medium_576/checkpoint_best_ema.pth

Starting checkpoint: /kaggle/input/notebooks/grigoriyyy/lct-rfdetr-medium-576/rfdetr_medium_576/checkpoint_best_ema.pth


In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model = RFDETRMedium(
    pretrain_weights=str(START_CKPT),
    resolution=RESOLUTION,
    num_classes=13,
)

t0 = time.perf_counter()

model.train(
    dataset_dir=str(RF_DATA),
    dataset_file="yolo",

    epochs=EPOCHS,
    resolution=RESOLUTION,

    batch_size="auto",
    grad_accum_steps=1,
    num_workers=4,

    output_dir=str(OUTPUT_DIR),
    seed=SEED,

    # Conservative fine-tune.
    # Previous RF-DETR default-scale experiments used ~1e-4 / 1.5e-4.
    # Start lower because weights are already adapted to our taxonomy/domain.
    lr=5e-5,
    lr_encoder=7.5e-5,

    use_ema=True,
    eval_base_model=False,

    checkpoint_interval=1,

    # With only four expensive epochs we want to observe all of them.
    early_stopping=False,
    skip_best_epochs=0,
    run_test=False,

    tensorboard=True,
    wandb=False,
    progress_bar="tqdm",
)

elapsed_h = (time.perf_counter() - t0) / 3600
print("Training hours:", elapsed_h)


[2026-09-25 11:02:19] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-25 11:02:19] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-25 11:02:26] [WARNING] rf-detr - [auto-batch] 1/509 trainable params received no gradient from the synthetic probe batch; their optimizer state is not modeled, so the returned batch size may be too permissive.
/usr/local/lib/python3.12/dist-packages/rfdetr/models/_assignment.py:76: RuntimeWarning: Triton linear-assignment support is unavailable for this CUDA input; using SciPy on CPU.
  assignment = torch_linear_assignment.batch_linear_assignment(stacked)


[2026-09-25 11:02:36] [INFO] rf-detr - [auto-batch] Applied EMA headroom (0.70): safe_micro_batch=3
[2026-09-25 11:02:36] [INFO] rf-detr - [auto-batch] device=Tesla T4 world_size=1 segmentation=False probe_resolution=800 max_targets_per_image=100
[2026-09-25 11:02:36] [INFO] rf-detr - [auto-batch] safe_micro_batch=3 grad_accum_steps=6 effective_batch_per_device=18 global_effective_batch=18
[2026-09-25 11:02:36] [INFO] rf-detr - [auto-batch] This probe estimates train-step-safe micro-batch only.
[2026-09-25 11:02:36] [INFO] rf-detr - [auto-batch] Validation/test (especially segmentation mask eval) may require more memory.
[2026-09-25 11:02:36] [INFO] rf-detr - [auto-batch] resolved train config: batch_size=3 grad_accum_steps=6 effective_batch_size=18


[2026-09-25 11:02:36] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-25 11:02:36] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[2026-09-25 11:02:52] [INFO] rf-detr - Using multi-scale training with square resize and scales: [800]
[2026-09-25 11:02:52] [INFO] rf-detr - Built 1 Albumentations transforms from config
[2026-09-25 11:02:52] [INFO] rf-detr - Built 1 Albumentations transforms from config
creating index...
index created!
[2026-09-25 11:04:31] [INFO] rf-detr - Using multi-scale training with square resize and scales: [800]
creating index...
index created!


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /kaggle/working/rfdetr_medium_640_finetune exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


...

[2026-09-25 14:24:43] [INFO] rf-detr - Best EMA metric improved to 0.6984 (epoch 1)


Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=4` reached.


[2026-09-25 17:46:22] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.0000, ema=0.6984)
Training hours: 6.73373543811


In [7]:
print("\nOutput files:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(OUTPUT_DIR))

print("\nIMPORTANT FILES")
for name in [
    "checkpoint_best_total.pth",
    "checkpoint_best_ema.pth",
    "checkpoint_best_regular.pth",
    "last.ckpt",
    "metrics.csv",
]:
    p = OUTPUT_DIR / name
    print(name, "->", p.exists(), p)



Output files:
  checkpoint_0.ckpt
  checkpoint_1.ckpt
  checkpoint_2.ckpt
  checkpoint_3.ckpt
  checkpoint_best_ema.pth
  checkpoint_best_total.pth
  events.out.tfevents.1790334172.14bfc1b7fea5.24.0
  hparams.yaml
  last_ema.pth
  metrics.csv
  training_config.json

IMPORTANT FILES
checkpoint_best_total.pth -> True /kaggle/working/rfdetr_medium_640_finetune/checkpoint_best_total.pth
checkpoint_best_ema.pth -> True /kaggle/working/rfdetr_medium_640_finetune/checkpoint_best_ema.pth
checkpoint_best_regular.pth -> False /kaggle/working/rfdetr_medium_640_finetune/checkpoint_best_regular.pth
last.ckpt -> False /kaggle/working/rfdetr_medium_640_finetune/last.ckpt
metrics.csv -> True /kaggle/working/rfdetr_medium_640_finetune/metrics.csv


После выполнения главное сохранить:
- `checkpoint_best_ema.pth`
- `checkpoint_best_total.pth`
- `last.ckpt`
- `metrics.csv`

Потом отдельно прогоняем этот best checkpoint на organizer100 тем же unified evaluator.
